In [ ]:
import os
import random
random.seed(42)
from time import time
import pickle
import numpy as np
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tools import Tools
from directories import Dicrectories
from scipy.stats import spearmanr

target_word_weight=defaultdict(list)
target_similarity=defaultdict(list)

def preprocess_text(text):
    return text

num = 0
clause_weight_threshold = 10
number_of_examples = 2000
accumulation = 25
clause_drop_p = 0.0
factor = 20
clauses = int(factor*20/(1.0 - clause_drop_p))
T = factor*40
s = 5.0

dataset_name = "rg-65"
dataset_dir = os.path.join("datasets", dataset_name)
files_start_name = os.path.join(dataset_dir, dataset_name)

print("Loading Vectorizer")
vectorizer_X = Tools.read_pickle_data("vectorizer_X.pickle")
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]
print("Loading Data")
X_train = Tools.read_pickle_data("X.pickle")
output_active, target_words = Tools.get_targets(files_start_name)
print(output_active)
pair_list = Tools.get_dataset_pairs(files_start_name + '.csv')
print(len(pair_list))
    

tm = TMAutoEncoder(clauses, T, s, output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=0.5)

print("\nAccuracy Over 40 Epochs:")
for e in range(1000):
    start_training = time()
    tm.fit(X_train, number_of_examples=number_of_examples)
    stop_training = time()
    total_time= stop_training-start_training
    print("\nEpoch #%d\n" % (e+1))
    print(f'epoch per time: {total_time}')

    profile = np.empty((len(target_words), clauses))
    for i in range(len(target_words)):
        weights = tm.get_weights(i)
        profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)

    similarity = cosine_similarity(profile)
    for i in range(len(target_words)):
        sorted_index = np.argsort(-1*similarity[i,:])
        for j in range(1, len(target_words)):
            target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]

    calculated_score=[]
    extracted_list = []
    original_score=[]
    word_pairs=[]
    for (x,y) in pair_list:
            if x in target_similarity:
                print("{} = {:.2f} - {}".format(x, target_similarity[x] * 10, y))
                word1_prof = target_similarity[x] * 10
                extracted_list.append((x, word1_prof))
                calculated_score.append(word1_prof)
                original_score.append(y)
                word_pairs.append(x)
    spearman_TM = spearmanr(original_score, calculated_score)
    spearman_TM = round(spearman_TM[0], 3)
    print(f'Spearman TM: {spearman_TM}')


In [ ]:
print("\n=====================================\nClauses\n=====================================")
for j in range(clauses):
    print("Clause #%-2d " % (j), end=' ')
    for tw in range(len(target_words)):
        print("%s:W%-5d " % (target_words[tw], tm.get_weight(tw, j)), end='| ')
    l = [] 
    number_of_literals = 0 
    for k in range(tm.clause_bank.number_of_literals):
        if tm.get_ta_action(j, k) == 1:
            number_of_literals = number_of_literals + 1
            if k < tm.clause_bank.number_of_features:
                l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
            else:
                l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
    print(": No of features:%-6d" % (number_of_literals), end=" ==> ")
    try:
        print(" - ".join(l))
    except UnicodeEncodeError:
        print(" exception ")
profile = np.empty((len(target_words), clauses))
for i in range(len(target_words)):
    weights = tm.get_weights(i)
    profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)

# output= open(files_start_name + '_knowledge_weights.pkl', "wb")
# pickle.dump(profile, output)
# output.close()

print("\n=====================================\nWord Similarity\n=====================================")
similarity = cosine_similarity(profile)
max_word_length = len(max(target_words, key=len))
list_of_words = []
target_words_with_min_max = []
for i in range(len(target_words)):
    row_of_similarity = []
    sorted_index = np.argsort(-1*similarity[i,:])
    min_similarity = 1.0
    max_similarity = 0.0
    word_similarity = []
    for j in range(1, len(target_words)):
        target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
        row_of_similarity.append(target_words[sorted_index[j]])
        word_similarity.append("{:<{}}({:.2f})  ".format(target_words[sorted_index[j]], max_word_length, similarity[i, sorted_index[j]]))
        if(min_similarity > similarity[i,sorted_index[j]]):
            min_similarity = similarity[i,sorted_index[j]]
        if(max_similarity < similarity[i,sorted_index[j]]):
            max_similarity = similarity[i,sorted_index[j]]

    output_line = f"{target_words[i]:<{max_word_length}}: Min:{min_similarity:.2f}, Max:{max_similarity:.2f}"
    print(output_line, end='     ==> ')
    print(word_similarity)
    list_of_words.append(row_of_similarity)
    target_words_with_min_max.append(output_line)